# 02 - Obtain Simulation Outputs

**Where this fits:** this is the boundary between the two halves of the
project. Everything before this point (`01_generate_samples.ipynb`) and after
it (`03_train_metamodel.ipynb` onward) is the Python/ML work in this repo. The
actual microsimulation itself — the R model that turns "here's a kit
allocation" into "here's how many overdose deaths we'd expect" — has its
source code included in this repo (`R_Files/`), but the Rhode-Island-specific
study data and calibration results that feed it are not, since those are
unpublished findings ahead of the associated paper (see the root README's
note on reproducibility). This notebook is the glue in between: it takes the
200,000 hypothetical allocations from notebook 01, which were run through
that simulation on a university HPC cluster, and combines the raw
per-allocation results into one clean dataset the neural network can train on.

## Overview
Each of the 200,000 Sobol-sampled naloxone allocation vectors generated in 
`01_generate_samples.ipynb` was passed as input to a high-fidelity stochastic 
microsimulation model of opioid overdose dynamics in Rhode Island.

## HPC Execution
Due to the stochastic nature of the simulation, each allocation vector was run 
across multiple random seeds to obtain stable estimates. Jobs were parallelized 
on a SLURM-based HPC cluster, with each job running a batch of allocation vectors 
independently.

**In plain terms:** the simulation models random, individual-level events
(who overdoses, whether naloxone is on hand, etc.), so running the *same*
allocation through it twice won't give identical results — one run might
predict 340 deaths, another 355, purely from randomness. Averaging across 100
different random seeds for each allocation smooths that noise out, so the
number the metamodel learns from reflects the allocation's real effect, not
the luck of one particular simulation run.

## Output Structure
Each simulation run produced a CSV file containing projected overdose death counts 
by city/town for a given allocation vector and seed.

## Combining Results
The code below aggregates the individual simulation outputs by averaging across 
seeds for each allocation vector, producing a single mean outcome per input vector.

**Note:** This notebook was executed on a university HPC cluster. The file paths
and SLURM configuration are specific to that environment and are provided here
for documentation purposes only.

In [ ]:
from pathlib import Path
import pandas as pd

# Aggregate simulation outputs across parameter seeds for each allocation vector
# For each iteration, 100 CSV files (one per seed) are averaged to produce
# a stable mean outcome per allocation vector

FOLDER_PATH = Path("/users/1/kuntz138/Final_Naloxone_Work")
N_ITERATIONS = 20
N_PARAMS = 100
ROWS_PER_ITERATION = 10000

simulation_dfs = []

for iteration in range(1, N_ITERATIONS + 1):
    
    # Load and concatenate all parameter seed files for this iteration
    file_paths = [FOLDER_PATH / f"naloxone_iteration_{iteration}_param_{i}.csv" 
                  for i in range(1, N_PARAMS + 1)]
    
    simulation_df = (pd.concat([pd.read_csv(f) for f in file_paths])
                       .reset_index(drop=True)
                       .iloc[:, 3:])  # Drop first three metadata columns
    
    # Average across parameter seeds for each allocation vector
    # Each allocation vector appears once per parameter file (N_PARAMS times total)
    outcome_col = simulation_df.columns[0]
    simulation_df[outcome_col] = (simulation_df[outcome_col]
                                  .values
                                  .reshape(N_PARAMS, ROWS_PER_ITERATION)
                                  .mean(axis=0)
                                  .repeat(N_PARAMS))
    
    simulation_dfs.append(simulation_df.iloc[:ROWS_PER_ITERATION])

# Combine all iterations into final dataset
final_simulation_df = pd.concat(simulation_dfs).reset_index(drop=True)
print(final_simulation_df.shape)

In [ ]:
# Save aggregated simulation input/output pairs for use in 03_train_metamodel.ipynb
BASE_DIR = Path.cwd().parents[0]
(BASE_DIR / 'data' / 'generated').mkdir(parents=True, exist_ok=True)
final_simulation_df.to_csv(BASE_DIR / 'data' / 'generated' / 'simulation_training_data.csv', index=False)